# 🔓 Solutions: Introduction to Machine Learning

> **Try first, peek second.** You learn far more by struggling with a problem for ten minutes than by reading its answer for ten seconds. Use these solutions to *compare* your approach, not to replace it. There is often more than one correct way.


In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor

### 🧩 Problem 1.1: Features and target in the churn data &nbsp; ⭐

Load `churn.csv`. Separate it into a **feature table `X`** and a **target vector `y`**, where the target is the `churned` column.

Afterwards: `X` must have **6 columns** and must **not** contain `churned`; `y` must be the `churned` column.

In [2]:
churn = load("churn.csv")

X = churn.drop(columns="churned")
y = churn["churned"]
print(X.shape, y.shape)

(1500, 6) (1500,)


In [3]:
# ✔ Self-check
check("X has 6 feature columns", lambda: X.shape == (1500, 6))
check("target is NOT inside X", lambda: "churned" not in X.columns)
check("y is the churned column", lambda: y.name == "churned" and len(y) == 1500)

✅ PASSED   X has 6 feature columns
✅ PASSED   target is NOT inside X
✅ PASSED   y is the churned column


True

**🧑‍🏫 Walkthrough.** `drop(columns=...)` keeps everything *except* the named column, so `X` has 7 − 1 = 6 columns. `y = churn["churned"]` is a pandas Series (a vector). Common slip: writing `churn[["churned"]]` with double brackets gives a 2-D *DataFrame* rather than a 1-D Series: many sklearn functions accept both, but it's good to know the difference.

### 🧩 Problem 1.2: Train, then infer &nbsp; ⭐

Using `students.csv`, train a `LinearRegression` that predicts `exam_score` from **`hours_studied`** only.

Then use it to predict the exam score for students who studied **2, 5 and 8 hours**, and store the three predictions (a NumPy array) in a variable named `preds`.

In [4]:
students = load("students.csv")

X = students[["hours_studied"]]
y = students["exam_score"]

model = LinearRegression().fit(X, y)                       # TRAINING

new = pd.DataFrame({"hours_studied": [2, 5, 8]})
preds = model.predict(new)                                  # INFERENCE
print(preds.round(1))

[43.  54.9 66.8]


In [5]:
# ✔ Self-check
check("three predictions", lambda: len(preds) == 3)
check("more study -> higher predicted score", lambda: preds[0] < preds[1] < preds[2])
check("5 hours predicts a score between 50 and 62", lambda: 50 < preds[1] < 62)

✅ PASSED   three predictions
✅ PASSED   more study -> higher predicted score
✅ PASSED   5 hours predicts a score between 50 and 62


True

**🧑‍🏫 Walkthrough.** `fit` runs once and learns `intercept_` and `coef_`; `predict` just plugs numbers into that learned formula. Notice the predictions rise by about the same amount per hour: that's the *linear* in linear regression. We never fit on the three new students: they are "unseen" data, exactly like real inference.

### 🧩 Problem 1.3: Parameter or hyperparameter? &nbsp; ⭐

Fill in the dictionary `answers` below. For each item write either `"parameter"` (learned from data by `fit`) or `"hyperparameter"` (chosen by a human before training).

In [6]:
answers = {
    "coef_ of LinearRegression": "parameter",
    "n_neighbors (k) in KNN": "hyperparameter",
    "learning rate": "hyperparameter",
    "weights of a neural network": "parameter",
    "alpha (penalty strength) in Ridge": "hyperparameter",
    "max_depth of a decision tree": "hyperparameter",
    "intercept_ of LinearRegression": "parameter",
    "number of clusters k in KMeans": "hyperparameter",
    "cluster centres found by KMeans": "parameter",
}

In [7]:
# ✔ Self-check
truth = {"coef_ of LinearRegression": "parameter", "n_neighbors (k) in KNN": "hyperparameter",
         "learning rate": "hyperparameter", "weights of a neural network": "parameter",
         "alpha (penalty strength) in Ridge": "hyperparameter", "max_depth of a decision tree": "hyperparameter",
         "intercept_ of LinearRegression": "parameter", "number of clusters k in KMeans": "hyperparameter",
         "cluster centres found by KMeans": "parameter"}
for item, correct in truth.items():
    check(item, lambda: answers[item] == correct)

✅ PASSED   coef_ of LinearRegression
✅ PASSED   n_neighbors (k) in KNN
✅ PASSED   learning rate
✅ PASSED   weights of a neural network
✅ PASSED   alpha (penalty strength) in Ridge
✅ PASSED   max_depth of a decision tree
✅ PASSED   intercept_ of LinearRegression
✅ PASSED   number of clusters k in KMeans
✅ PASSED   cluster centres found by KMeans


**🧑‍🏫 Walkthrough.** The KMeans pair is the classic trap: **how many** clusters (*k*) is a decision *you* make → hyperparameter; **where** the cluster centres end up is *learned* → parameter. Same algorithm, one of each.

### 🧩 Problem 1.4: Discover the hidden rule &nbsp; ⭐⭐

A mysterious process generates a target from two inputs. The data is created for you below. Train a `LinearRegression` on it and report the learned numbers:

* `w1`: the coefficient of the first input
* `w2`: the coefficient of the second input
* `b`: the intercept

The true rule is a secret, but after training, your numbers should reveal it. *(Run the cell and read your answer off the learned parameters!)*

In [8]:
rng = np.random.default_rng(7)
X_hidden = rng.uniform(-5, 5, size=(200, 2))
y_hidden = 3 * X_hidden[:, 0] - 2 * X_hidden[:, 1] + 5 + rng.normal(0, 0.5, 200)

model = LinearRegression().fit(X_hidden, y_hidden)
w1, w2 = model.coef_
b = model.intercept_
print(f"My discovered rule: y = {w1:.2f}·x1 + ({w2:.2f})·x2 + {b:.2f}")

My discovered rule: y = 3.00·x1 + (-2.02)·x2 + 4.90


In [9]:
# ✔ Self-check
check("w1 is close to the true value", lambda: abs(w1 - 3) < 0.2)
check("w2 is close to the true value", lambda: abs(w2 + 2) < 0.2)
check("b is close to the true value",  lambda: abs(b - 5) < 0.3)

✅ PASSED   w1 is close to the true value
✅ PASSED   w2 is close to the true value
✅ PASSED   b is close to the true value


True

**🧑‍🏫 Walkthrough.** We never told the model the numbers 3, −2 and 5: it recovered them (approximately, because of the noise) purely from examples. This is the *Celsius → Fahrenheit* trick again, but with two inputs and noise. Real-world data is rarely this clean, which is why we'll need careful evaluation in later lessons.

### 🧩 Problem 1.5: Name that learning problem &nbsp; ⭐⭐

Label each scenario with **exactly one** of these strings:
`"supervised-regression"`, `"supervised-classification"`, `"unsupervised-clustering"`, `"unsupervised-dimensionality-reduction"`, `"reinforcement"`.

Complete the dictionary `scenarios`.

In [10]:
scenarios = {
    "Predict tomorrow's electricity demand (in MW) from the weather forecast": "supervised-regression",
    "Flag a card transaction as fraud, using past transactions labelled fraud/not-fraud": "supervised-classification",
    "Group news articles into topics when nobody has assigned topics": "unsupervised-clustering",
    "Compress 1,000 sensor readings per machine into 10 summary numbers": "unsupervised-dimensionality-reduction",
    "Teach a robot to walk by rewarding forward progress": "reinforcement",
    "Name the species of a flower from its petal measurements (species known for training flowers)": "supervised-classification",
}

In [11]:
# ✔ Self-check
truth = ["supervised-regression", "supervised-classification", "unsupervised-clustering",
         "unsupervised-dimensionality-reduction", "reinforcement", "supervised-classification"]
for (scenario, given), correct in zip(scenarios.items(), truth):
    check(scenario[:60] + "...", lambda: given == correct)

✅ PASSED   Predict tomorrow's electricity demand (in MW) from the weath...
✅ PASSED   Flag a card transaction as fraud, using past transactions la...
✅ PASSED   Group news articles into topics when nobody has assigned top...
✅ PASSED   Compress 1,000 sensor readings per machine into 10 summary n...
✅ PASSED   Teach a robot to walk by rewarding forward progress...
✅ PASSED   Name the species of a flower from its petal measurements (sp...


**🧑‍🏫 Walkthrough.** The decision procedure: **labels?** no → unsupervised (or reinforcement if there is an agent and rewards). **Labels?** yes → look at the *type* of the label: number → regression, category → classification. "MW of demand" is a number; "fraud / not fraud" and "species" are categories.

### 🧩 Problem 1.6: Turn the knob: hyperparameters in action &nbsp; ⭐⭐⭐

Train four `KNeighborsRegressor` models on `students.csv` (feature: `hours_studied`, target: `exam_score`) using **k = 1, 5, 25, 100**.

For each, predict the score for a student who studied **6.0 hours**. Store the results in a dictionary `k_preds` like `{1: 61.2, 5: ...}`.

Then print the **range** (max − min) of the four predictions. *Bonus question to ponder:* why do the predictions for large k look more alike?

In [12]:
students = load("students.csv")
X, y = students[["hours_studied"]], students["exam_score"]

k_preds = {}
for k in [1, 5, 25, 100]:
    knn = KNeighborsRegressor(n_neighbors=k).fit(X, y)
    k_preds[k] = knn.predict(pd.DataFrame({"hours_studied": [6.0]}))[0]

print({k: round(v, 1) for k, v in k_preds.items()})
print("range:", round(max(k_preds.values()) - min(k_preds.values()), 1))

{1: np.float64(67.3), 5: np.float64(59.3), 25: np.float64(58.7), 100: np.float64(58.8)}
range: 8.6


In [13]:
# ✔ Self-check
check("four predictions stored", lambda: sorted(k_preds) == [1, 5, 25, 100])
check("all predictions are plausible scores", lambda: all(40 < v < 80 for v in k_preds.values()))
check("k=1 and k=100 disagree by at least a point", lambda: abs(k_preds[1] - k_preds[100]) > 1)

✅ PASSED   four predictions stored
✅ PASSED   all predictions are plausible scores
✅ PASSED   k=1 and k=100 disagree by at least a point


True

**🧑‍🏫 Walkthrough.** Small *k* → the prediction hangs on a couple of neighbouring students (so it's noisy). Large *k* → the prediction is an average over so many students that it becomes smooth, and for k=100 it drifts toward the dataset's overall behaviour. A hyperparameter like *k* controls the model's **flexibility**; finding the right amount of flexibility is the core of Lesson 5.